# Having Fun with CMOR

*pyku* provides functions to CMORize data, automatically handling a significant number of attributes and unit conversions. However, global attributes must be manually set, as they cannot be inferred. The first step, as usual, is to load the xarray library and the pyku extension.

In [ ]:
import xarray as xr
import pyku

## DRS definition

*pyku* includes DRS definition files. You can display the full list of implemented CMOR and CMOR-like DRS definitions as follows:

In [ ]:
pyku.list_drs_standards()

In this tutorial, we will CMORize COSMO-CLM data using the CORDEX CMOR standard. Behind the scenes, pyku utilizes YAML metadata files, which are structured like this:

````
    cordex:

        # References
        # ----------

        # http://is-enes-data.github.io/
        # http://is-enes-data.github.io/cordex_archive_specifications.pdf
        # http://is-enes-data.github.io/CORDEX_variables_requirement_table.pdf

        # The stem/parent wording stems from the pathlib library
        # https://docs.python.org/3/library/pathlib.html

        metadata:
            - CORDEX_domain
            - driving_model_id
            - driving_experiment_name
            - driving_model_ensemble_member
            - model_id
            - rcm_version_id
            - frequency
            - product
            - institute_id
            - experiment_id

        stem_pattern:
            "{variable_name}_{CORDEX_domain}_{driving_model_id}_\
             {driving_experiment_name}_{driving_model_ensemble_member}_\
             {model_id}_{rcm_version_id}_{frequency}_{start_time}-{end_time}"

        parent_pattern:
            "{product}/{CORDEX_domain}/{institute_id}/{driving_model_id}/\
             {experiment_id}/{driving_model_ensemble_member}/{model_id}/\
             {rcm_version_id}/{frequency}"
````

## Get exemplary ICON data

Get ICON exemplary data, which consists of raw model output without any post-processing.

In [ ]:
# List files using UNIX file pattern
# ----------------------------------

grib_files = pyku.resources.get_test_data('icon_grib_files')
grid_file = pyku.resources.get_test_data('icon_grid_file')

## Exploring the dataset

We open the dataset using standard xarray options. It is important to specify how xarray should handle variables when working with multiple files. By default, georeferencing metadata (like the CRS attribute) is tied to the time dimension. Here, we instruct xarray to treat the CRS as a single variable across all files, independent of dataset dimensions, if it remains the same. Additionally, for this demonstration, we only use a subset of the data to ensure faster execution. In the background, both xarray and pyku are optimized for handling large datasets, allowing terabytes of data to be lazily loaded and processed on demand.

In [ ]:
ds = xr.open_mfdataset(
    grib_files,
    engine='cfgrib',
    combine='nested',
    compat='no_conflicts',
    coords='different',
    time_dims=["valid_time"]
)

ds

## Modify the data

This preprocessing step standardizes the raw GRIB data for compatibility with pyku. Specifically, we rename the `valid_time` dimension to `time` and drop the original model-run initialization `time` and the forecast time `step`. Since the dataset is strictly 2D, the `heightAboveGround` coordinate is removed to simplify the data structure. Finally, we rename the data variable to cell, which is a prerequisite for correctly mapping the geographic coordinate system in subsequent steps.

In [ ]:
ds = ds.drop_vars(['heightAboveGround'])
ds = ds.rename({'valid_time': 'time'})
ds = ds.rename_dims({'values': 'cell'})

## Attach the georeferencing

ICON model output is typically stored on an unstructured grid, where meteorological variables are separated from their spatial coordinates. To perform georeferencing, we must retrieve a standalone grid file (external coordinates) containing the latitude and longitude for each cell center. This grid information is essential for mapping the raw data to a geographic coordinate system.

In [ ]:
grid = xr.open_dataset(
    pyku.resources.get_test_data('icon_grid_file')
)

We can merge the grid cell center latitudes and longitudes to the data

In [ ]:
ds = xr.merge([
    ds,
    grid[['clon', 'clat']]
])

## Regridding the data

With the coordinates attached, we can now apply one of pyku’s pre-defined geographic projections. Note that pyku does not currently support spatial projections for dask-chunked data on unstructured grids; datasets must be loaded into memory before the transformation.

In [ ]:
ds = ds.chunk(chunks=-1).pyku.project('GER-0275')

In [ ]:
ds.isel(time=0).ana.one_map(var='t2m', crs='GER-0275')

## Selecting variables for CMORization

*pyku* can select a single variable as a dataset, with all global attributes kept, and perform the CMORization process. It is important to note that CMORization should be done one variable at a time. To list all available georeferenced data that can be CMORized (excluding non-data variables like the CRS or time bounds), you can use the following pyku functionality.

In [ ]:
ds.pyku.get_geodata_varnames()

From this list, you can select a specific variable while preserving all its attributes. Special variables, such as the CRS and time bounds, are automatically handled.

## Setting CMOR attibutes and names

The dataset is now ready for CMORization. Note that all variable attributes and conventions are applied, except for the global metadata, which still need to be set.

In [ ]:
cmorized = ds.pyku.get_geodataset('t2m').pyku.cmorize()
cmorized

For verification, run a DRS check using the following command:

In [ ]:
cmorized.pyku.check_drs(standard='cordex')

## Setting global metadata

Running a sanity check to verify compliance with the standard, we confirm that the global metadata have not been set, as they cannot be inferred automatically. The global metadata must be manually provided to the cmorize function in the form of a dictionary. For operational use, it is recommended to define this metadata in a YAML file, which can then be loaded into a dictionary

In [ ]:
global_metadata = {
    'title': (
        "COSMO_CLM_5.00_clm16 simulation (0.0275 Deg) with ERA5T (direct nest, since "
        "2022) forcing, domain Germany, FPS-C tuned setup"
    ),
    'CORDEX_domain': 'GER-0275',
    'driving_model_id': 'ECMWF-ERA5T',
    'driving_experiment_name': 'evaluation',
    'experiment_id': 'evaluation',
    'driving_experiment': 'ECMWF-ERA5T, evaluation, r1i1p1',
    'driving_model_ensemble_member': 'r1i1p1',
    'experiment': 'evaluation run with ECMWF-ERA5T reanalysis forcing',
    'model_id': 'CLMcom-DWD-CCLM5-0-16',
    'institute_id': 'CLMcom-DWD',
    'institution': (
        "DWD (Deutscher Wetterdienst, Offenbach, Germany) in collaboration with the "
        "CLM-Community"
    ),
    'rcm_version_id': 'x0n1-v1',
    'contact': 'klima.offenbach@dwd.de',
    'nesting_levels': '1',
    'comment_nesting': 'these are results of a direct downscaling (one nest) approach',
    'comment_1stNest': (
        "convection permitting simulation driven by ECMWF-ERA5T (direct downscaling) "
        "comment_2ndNest: not used "
    ),
    'comment': (
        "Convection-permitting evaluation run for Germany (HoKliSim-DeT) performed by "
        "DWD Offenbach. Configuration adapted from the CORDEX FPS Convection setup, "
        "which was developed and tuned by the CRCS working group of the CLM Community. "
        "Computing system: Aurora NEC at DWD. The Deutscher Wetterdienst (DWD) is the "
        "producer of the data. The General Terms and Conditions of Business and "
        "Delivery apply for services provided by DWD "
        "(http://www.dwd.de/EN/service/terms/terms.html)."
    ),
    "rcm_config_cclm": "GER-0275_CLMcom-DWD-CCLM5-0-16_config",
    "rcm_config_int2lm": "GER-0275_CLMcom-DWD-INT2LM2-0-6-modif_config",
    "source": "Climate Limited-area Modelling Community (CLM-Community)",
    "references": "http://www.clm-community.eu, http://www.dwd.de",
    "product": "output",
    "project_id": "DWD-CPS",
}

Now, we CMORize the data again, this time setting the global attributes for CMOR compliance. Afterward, we perform another sanity check and observe that the global attributes have been correctly set.

In [ ]:
cmorized = ds.pyku.get_geodataset('t2m')
cmorized = cmorized.pyku.cmorize(global_metadata=global_metadata)
cmorized.pyku.check_drs(standard='cordex')

An additional sanity check can be performed using the pyku check function to identify any potential issues:

In [ ]:
cmorized.pyku.check_metadata(standard='cordex')

## CMORized data filename

From here, we can verify the name that the final data file should have.

In [ ]:
cmorized.pyku.drs_filename(standard='cordex')

## Geographic projections

The data can be reprojected as needed, with all available CMOR metadata set automatically. Note how the grid has been modified in the metadata to comply with the CMOR standard.

In [ ]:
cmorized = ds.pyku.get_geodataset('t2m')
cmorized = cmorized.pyku.cmorize(global_metadata=global_metadata)
cmorized = cmorized.pyku.project('HYR-LAEA-5')
cmorized.pyku.check_drs(standard='cordex')

The filename should now be as follows:

In [ ]:
cmorized.pyku.drs_filename(standard='cordex')

## Resampling data

Similarly, we can transform the data through resampling.

In [ ]:
cmorized = ds.pyku.get_geodataset('t2m')
cmorized = cmorized.pyku.project('HYR-LAEA-5')
cmorized = cmorized.pyku.resample_datetimes(frequency='1D', how='mean')
cmorized = cmorized.pyku.cmorize(global_metadata=global_metadata)
cmorized

In [ ]:
cmorized.time_bnds

Again, note that the metadata have been set in accordance with the CMOR standard during this operation.

In [ ]:
cmorized.pyku.check_drs(standard='cordex')

Pay particular attention to how the variable attributes and the ``cell_methods`` have been set:

In [ ]:
cmorized.tas.attrs

## Writing NetCDF data

The final step is to write the data, which can be quite large, into packets that conform to the CMOR standard. Specifically, this means:

* Hourly data will be split into packets of one year.
* Three-hourly data will be split into packets of one year.
* Six-hourly data will be split into packets of one year.
* Twelve-hourly data will be split into packets of five years.
* Daily data will be split into packets of five years.
* Monthly data will be split into packets of ten years.
* Seasonal data will be split into packets of ten years.
* Yearly data and above will be split into packets of one hundred years.

For this demonstration, we are using only a subset of data to ensure the tutorial runs in a timely manner, rather than showcasing the full capabilities. However, this functionality is built-in and will operate automatically for large datasets without requiring user input. First, we create a temporary directory to store the data. This directory will be deleted at the end of this session.

In [ ]:
import tempfile

temp_dir = tempfile.TemporaryDirectory()
print("Temporary directory:", temp_dir.name)

In [ ]:
pyku.drs.to_drs_netcdfs(cmorized, base_dir=temp_dir.name, standard='cordex')

In [ ]:
!find {temp_dir.name} -type f

In [ ]:
file = !find {temp_dir.name} -type f
!ncdump -h {file[0]}

And finally cleanup the temporary directory

In [ ]:
temp_dir.cleanup()